# Fairness Audit — Dev Log

## Objetivo e papel no pipeline

`core/fairness_audit` é o primeiro módulo da **Onda 1 do V2**: um motor
estatístico determinístico (sem ML/treinamento) que audita equidade de
decisões de IA em relação a um atributo protegido (gênero, raça
autodeclarada, faixa etária, etc.), usando duas métricas padrão da
literatura de fairness em ML.

**Por que isso importa em governança de IA/LGPD**: a LGPD (Art. 20) já trata
de decisões automatizadas e o direito à revisão; auditoria de equidade é o
complemento natural — não basta a decisão ser "explicável" (`explainability`,
V1), ela também precisa ser **equitativa** entre grupos.

## Decisões de design

### Regra dos 80% (disparate impact) + diferença de paridade demográfica

Duas métricas complementares, ambas padrão de mercado (a regra dos 80% é
usada pela EEOC americana e citada em auditorias de IA no Brasil também):

- **Disparate impact ratio**: `taxa_grupo / taxa_grupo_referência`. Passa se
  `>= threshold` (default 0.8).
- **Demographic parity difference**: diferença absoluta de taxas. Passa se
  `<= (1 - threshold)`.

### Grupo de referência automático

O grupo de maior taxa de seleção vira a referência — convenção padrão da
regra dos 80% (compara-se sempre contra o grupo mais favorecido), evitando
que o chamador precise escolher manualmente qual grupo é "o padrão".

### Limitação documentada, não escondida

Com 3+ grupos, a comparação é sempre grupo-a-grupo contra a referência, não
par-a-par entre todos os grupos — suficiente para a regra dos 80%, mas não
substitui uma análise multivariada completa. E não há teste de significância
estatística (intervalo de confiança) sobre as diferenças — amostras pequenas
podem gerar `disparate_impact` ruidoso. Ver `CHANGELOG.md` para a lista
completa.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.fairness_audit.engine import audit_fairness

# Cenário: sistema de aprovação de crédito por IA, auditado por gênero.
records = (
    [{"decision": "approved", "gender": "F"} for _ in range(38)]
    + [{"decision": "denied", "gender": "F"} for _ in range(62)]
    + [{"decision": "approved", "gender": "M"} for _ in range(55)]
    + [{"decision": "denied", "gender": "M"} for _ in range(45)]
)
result = audit_fairness(records, outcome_key="decision", protected_attribute_key="gender", favorable_outcome="approved")
print("Taxas de seleção:", result.selection_rates)
print("Grupo de referência:", result.reference_group)
print("Justo (overall_fair)?", result.overall_fair)
for m in result.metrics:
    print(f" - {m.metric_name} | grupo={m.group} | valor={m.value} | limiar={m.threshold} | passou={m.passed}")
print()
print(result.summary)

Taxas de seleção: {'F': 0.38, 'M': 0.55}
Grupo de referência: M
Justo (overall_fair)? False
 - disparate_impact_ratio | grupo=F | valor=0.6909 | limiar=0.8 | passou=False
 - demographic_parity_difference | grupo=F | valor=0.17 | limiar=0.2 | passou=True

Auditoria de equidade sobre 'gender' (200 registros, 2 grupo(s)): COM indícios de disparidade em relação ao grupo de referência 'M' (55.0% de taxa de seleção). Taxas de seleção por grupo: F (38.0%), M (55.0%).


Neste cenário (dados fictícios para demonstração), o `disparate_impact_ratio`
de 0.69 fica abaixo do limiar de 0.8 — sinaliza indício real de disparidade
entre os grupos, exatamente o comportamento que a regra dos 80% pretende
capturar.

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/fairness_audit/tests -v
```

10 testes cobrindo: taxas iguais (justo), disparidade real (não-justo), grupo
de referência correto, 3+ grupos, grupo único (justo por vacuidade),
`favorable_outcome`/`threshold` customizados, validação de entrada, proteção
contra divisão por zero.

## Handoff Summary

- **Status:** ✅ done — 10/10 testes passando.
- **Consumível por:** qualquer sistema de decisão do AthenaGov AI (via
  injeção de `records`) — ex. um futuro relatório agregado do
  `governance_copilot` sobre decisões de política acumuladas.
- **Limitações (TODO onda futura):** significância estatística, comparação
  par-a-par completa entre 3+ grupos, controle de variáveis confundidoras.